# Séance 10 · Exercices — Kaggle Titanic (1/2) : la même démarche, un autre vaisseau · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐ Intermédiaire** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

En séance, tu as exploré le Titanic, préparé ses données et fait ta soumission sur Kaggle : **la soumission Kaggle, elle, reste dans la leçon**. Ici, tu refais toute la démarche, seul·e, sur **d'autres données** : celles du *Spaceship Titanic*, la compétition d'entraînement que Kaggle propose après le Titanic. Les notions sont les mêmes (explorer, vérifier des hypothèses, remplir les cases vides, extraire des variables avec une regex, écrire `preparer()`, mesurer un modèle, fabriquer un fichier de soumission), mais les colonnes, les pièges et les bonnes idées sont nouveaux.

**Comment travailler** : lis l'énoncé, code dans la cellule `# À toi`, lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab** (rien à installer). Exécute chaque cellule avec `Maj + Entrée`, dans l'ordre : les exercices réutilisent ce qui précède.

## Quiz d'ouverture (2 min)

Réponds dans ta tête, puis déplie la réponse. Ce sont les questions qu'on se pose à voix haute au début de la séance.

**1. Pourquoi transformer « male » / « female » en 0 / 1 ?**
- a. Un modèle ne comprend que des nombres
- b. Pour économiser de la mémoire
- c. C'est une règle de Kaggle
- d. Pour anonymiser les passagers

<details><summary>Réponse</summary>

**a. Un modèle ne comprend que des nombres** — scikit-learn fait des calculs. Un mot n'est pas un nombre, il faut l'encoder.

</details>

**2. Quel fichier dépose-t-on sur Kaggle pour être classé ?**
- a. Le notebook
- b. train.csv
- c. submission.csv, avec une prédiction par passager de test.csv
- d. Une capture d'écran du score

<details><summary>Réponse</summary>

**c. submission.csv, avec une prédiction par passager de test.csv** — Deux colonnes : PassengerId et Survived. Kaggle compare avec les vraies réponses qu'il garde secrètes.

</details>

## Préparation

**L'histoire (inventée par Kaggle).** En l'an 2912, le paquebot spatial *Spaceship Titanic* emmène 13 000 passagers vers trois exoplanètes. Il frôle une anomalie de l'espace-temps : près d'un passager sur deux est **transporté dans une autre dimension**. Ton travail : prédire qui a été transporté. Les données sont **fictives** (fabriquées par Kaggle pour s'entraîner), mais elles se comportent comme de vraies données : des cases vides partout, du texte à transformer en nombres, et des indices cachés dans les identifiants.

| Colonne | Signification |
|---|---|
| `PassengerId` | `gggg_pp` : le numéro du groupe qui voyage ensemble, puis le numéro dans le groupe |
| `HomePlanet` | planète de départ : `Earth`, `Europa`, `Mars` |
| `CryoSleep` | `True` si le passager a voyagé endormi dans sa cabine (sommeil cryogénique) |
| `Cabin` | `pont/numéro/côté`, par exemple `B/0/P` ; côté `P` = bâbord, `S` = tribord |
| `Destination` | l'exoplanète visée |
| `Age`, `VIP` | l'âge, et `True` pour un service VIP payé |
| `RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck` | ce que le passager a dépensé à bord, poste par poste |
| `Name` | prénom et nom |
| `Transported` | **la cible** : `True` si le passager a été transporté |

La cellule charge les 8 693 passagers de `train.csv` (deux copies publiques identiques au fichier de Kaggle, figées à une version précise) et met de côté un **faux test** pour l'exercice 11 : les groupes numérotés 8500 et plus, sans la colonne `Transported`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# train.csv de la compétition Spaceship Titanic, deux copies publiques identiques, figées à une version
URLS = [
    "https://raw.githubusercontent.com/AmirMAAI/Spaceship_Titanic_dataset/a277ca40a73ffa2e83e477354cea20625e6470d2/spaceship-titanic_train.csv",
    "https://huggingface.co/datasets/Hugo0133/Spaceship-Titanic/resolve/dfc1e1d39a3193cfe99c53a51533b9ce04c71fb9/train.csv",
]


def charger(urls):
    """Essaie chaque copie l'une après l'autre ; s'arrête à la première qui répond."""
    for url in urls:
        try:
            return pd.read_csv(url)
        except Exception as erreur:
            print("Copie injoignable, on essaie la suivante :", erreur)
    raise RuntimeError("Aucune copie ne répond : vérifie la connexion internet, puis relance la cellule.")


vaisseau = charger(URLS)
print(vaisseau.shape[0], "passagers,", vaisseau.shape[1], "colonnes")

# Le faux test de l'exercice 11 : des groupes entiers, comme le vrai test.csv de Kaggle
numero_groupe = vaisseau["PassengerId"].str[:4].astype(int)
faux_test = vaisseau[numero_groupe >= 8500].drop(columns=["Transported"])
reponses_cachees = vaisseau.loc[numero_groupe >= 8500, "Transported"].astype(int)
vaisseau_reduit = vaisseau[numero_groupe < 8500]


# --- Vérification automatique : affiche ✅ ou ❌, ne plante jamais ---
def verifier(nom, condition):
    """condition : un booléen, ou une fonction sans argument qui renvoie un booléen."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))


def proche(valeur, attendu, tolerance=0.01):
    """Vrai si valeur est un nombre à moins de `tolerance` de attendu."""
    try:
        return abs(float(valeur) - attendu) <= tolerance
    except (TypeError, ValueError):
        return False

print("Prêt !", len(vaisseau_reduit), "passagers pour apprendre,", len(faux_test), "dans le faux test.")

## Exercice 1 ⭐ · Premier coup d'œil

Avant de modéliser, on regarde le tableau `vaisseau`. Remplis les 4 variables :
- `nb_passagers` : nombre de lignes
- `nb_colonnes` : nombre de colonnes
- `nb_ages_manquants` : nombre de cases vides dans la colonne `Age`
- `colonne_la_plus_vide` : le nom de la colonne qui a le plus de cases vides (une chaîne de caractères)

Résultat attendu : `8693 passagers, 14 colonnes, 179 âges manquants`. Contrairement au Titanic, aucune colonne n'est vide aux trois quarts : il manque un peu de tout, partout.

<details><summary>Indice</summary>

`vaisseau.shape` donne (lignes, colonnes). `vaisseau.isna().sum()` compte les vides par colonne, et `.idxmax()` donne le nom de la plus grande valeur.

</details>

In [ ]:
# À toi
nb_passagers = None
nb_colonnes = None
nb_ages_manquants = None
colonne_la_plus_vide = None

print(nb_passagers, "passagers,", nb_colonnes, "colonnes,", nb_ages_manquants, "âges manquants")
print("La colonne la plus vide :", colonne_la_plus_vide)

In [ ]:
verifier("Exercice 1 · lignes et colonnes", lambda: (nb_passagers, nb_colonnes) == (8693, 14))
verifier("Exercice 1 · âges manquants", lambda: nb_ages_manquants == 179)
verifier("Exercice 1 · colonne la plus vide", lambda: colonne_la_plus_vide == vaisseau.isna().sum().idxmax())

<details><summary>Solution</summary>

```python
nb_passagers, nb_colonnes = vaisseau.shape
nb_ages_manquants = vaisseau["Age"].isna().sum()
colonne_la_plus_vide = vaisseau.isna().sum().idxmax()

print(nb_passagers, "passagers,", nb_colonnes, "colonnes,", nb_ages_manquants, "âges manquants")
print("La colonne la plus vide :", colonne_la_plus_vide)   # CryoSleep : 217 cases vides
print(vaisseau.isna().sum())
```

</details>

## Exercice 2 ⭐ · Compter avec value_counts

Trois comptes pour faire connaissance avec les passagers :
- `nb_terriens` : le nombre de passagers partis de la Terre (`HomePlanet` vaut `"Earth"`)
- `destination_favorite` : la destination la plus fréquente (une chaîne de caractères)
- `taux_transportes` : la part de passagers transportés, entre 0 et 1

Résultat attendu : `4602` Terriens, et un taux de transportés d'environ `0.50`. Un passager sur deux : prédire « personne n'est transporté » ne fait donc pas mieux que pile ou face.

<details><summary>Indice</summary>

`vaisseau["HomePlanet"].value_counts()` donne un compte par planète, et `["Earth"]` lit celui de la Terre. `.idxmax()` donne l'étiquette du plus grand compte. `Transported` vaut `True` ou `False` : sa moyenne (`.mean()`) est la part de `True`.

</details>

In [ ]:
# À toi
nb_terriens = None
destination_favorite = None
taux_transportes = None

print(nb_terriens, "Terriens · destination favorite :", destination_favorite, "· transportés :", taux_transportes)

In [ ]:
verifier("Exercice 2 · Terriens", lambda: nb_terriens == 4602)
verifier("Exercice 2 · destination favorite", lambda: destination_favorite == "TRAPPIST-1e")
verifier("Exercice 2 · taux de transportés ≈ 0.50", lambda: proche(taux_transportes, 0.5036, 0.001))

<details><summary>Solution</summary>

```python
nb_terriens = vaisseau["HomePlanet"].value_counts()["Earth"]
destination_favorite = vaisseau["Destination"].value_counts().idxmax()
taux_transportes = vaisseau["Transported"].mean()

print(vaisseau["HomePlanet"].value_counts())
print(vaisseau["Destination"].value_counts())
print(nb_terriens, "Terriens · destination favorite :", destination_favorite, "· transportés :", round(taux_transportes, 3))
```

</details>

## Exercice 3 ⭐ · Hypothèse : « les dormeurs ont été emportés »

Un passager en sommeil cryogénique reste enfermé dans sa cabine pendant tout le voyage. Hypothèse : l'anomalie a surtout frappé les cabines des dormeurs. Vérifie-la :
- `taux_dormeurs` : la part de transportés parmi les passagers qui ont `CryoSleep` à `True`
- `taux_eveilles` : la même part parmi ceux qui ont `CryoSleep` à `False`

Résultat attendu : environ `0.82` contre `0.33`. C'est l'équivalent de « les femmes d'abord » sur le Titanic : l'indice le plus fort du tableau.

<details><summary>Indice</summary>

`vaisseau.groupby("CryoSleep")["Transported"].mean()` donne les deux taux d'un coup (les cases vides de `CryoSleep` sont laissées de côté) : lis-les avec `[True]` et `[False]`.

</details>

In [ ]:
# À toi
taux_dormeurs = None
taux_eveilles = None

print("Dormeurs :", taux_dormeurs, "- éveillés :", taux_eveilles)

In [ ]:
verifier("Exercice 3 · dormeurs ≈ 0.82", lambda: proche(taux_dormeurs, 0.8176, 0.001))
verifier("Exercice 3 · éveillés ≈ 0.33", lambda: proche(taux_eveilles, 0.3289, 0.001))

<details><summary>Solution</summary>

```python
par_sommeil = vaisseau.groupby("CryoSleep")["Transported"].mean()
taux_dormeurs = par_sommeil[True]
taux_eveilles = par_sommeil[False]

print("Dormeurs :", round(taux_dormeurs, 3), "- éveillés :", round(taux_eveilles, 3))
par_sommeil.plot(kind="bar", color="tab:purple", figsize=(5, 3.5), title="Part de transportés selon le sommeil")
plt.xticks(rotation=0)
plt.show()
# 82 % contre 33 % : dormir dans sa cabine multiplie par 2,5 le risque d'être transporté.
```

</details>

## Exercice 4 ⭐ · Hypothèse : ceux qui dépensent

Un passager éveillé va au restaurant, au spa, au salon de réalité virtuelle. Crée une nouvelle colonne, puis vérifie l'hypothèse « ceux qui n'ont rien dépensé ont plus souvent été transportés » :
- `vaisseau["Depense"]` : la somme des 5 colonnes de dépenses (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`)
- `taux_sans_depense` : la part de transportés parmi ceux dont `Depense` vaut 0
- `taux_avec_depense` : la même part parmi ceux qui ont dépensé quelque chose

Résultat attendu : environ `0.79` contre `0.30`, sur 3 653 passagers qui n'ont rien dépensé.

<details><summary>Indice</summary>

`vaisseau[liste_de_colonnes].sum(axis=1)` additionne les colonnes ligne par ligne ; une case vide y compte pour 0. Puis filtre avec `vaisseau["Depense"] == 0` et `vaisseau["Depense"] > 0`.

</details>

In [ ]:
# À toi
DEPENSES = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
taux_sans_depense = None
taux_avec_depense = None

print("Rien dépensé :", taux_sans_depense, "- a dépensé :", taux_avec_depense)

In [ ]:
verifier("Exercice 4 · la colonne Depense existe", lambda: proche(vaisseau["Depense"].sum(), vaisseau[DEPENSES].sum().sum(), 0.5))
verifier("Exercice 4 · rien dépensé ≈ 0.79", lambda: proche(taux_sans_depense, 0.7865, 0.001))
verifier("Exercice 4 · a dépensé ≈ 0.30", lambda: proche(taux_avec_depense, 0.2986, 0.001))

<details><summary>Solution</summary>

```python
DEPENSES = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
vaisseau["Depense"] = vaisseau[DEPENSES].sum(axis=1)
taux_sans_depense = vaisseau.loc[vaisseau["Depense"] == 0, "Transported"].mean()
taux_avec_depense = vaisseau.loc[vaisseau["Depense"] > 0, "Transported"].mean()

print((vaisseau["Depense"] == 0).sum(), "passagers n'ont rien dépensé")
print("Rien dépensé :", round(taux_sans_depense, 3), "- a dépensé :", round(taux_avec_depense, 3))
```

Les deux hypothèses se ressemblent beaucoup : l'exercice suivant montre pourquoi.

</details>

## Exercice 5 ⭐⭐ · Remplir les cases vides avec un raisonnement

Dans la leçon, l'âge manquant d'un passager venait de son titre. Ici, deux raisonnements du même genre :

1. Un dormeur reste dans sa cabine : il ne peut rien dépenser. Vérifie-le avec `nb_dormeurs_depensiers`, le nombre de passagers qui ont `CryoSleep` à `True` **et** `Depense` plus grand que 0.
2. On peut donc **déduire** une partie des `CryoSleep` vides : qui a dépensé était éveillé. `nb_deduits_eveilles` : le nombre de passagers dont `CryoSleep` est vide et `Depense` plus grand que 0.
3. `cryo_rempli` : la colonne `CryoSleep` en 0 / 1, sans case vide. Une case vide devient 0 si le passager a dépensé, 1 sinon (parmi ceux qui n'ont rien dépensé, 85 % dorment).
4. `age_rempli` : la colonne `Age`, où chaque âge manquant prend l'âge **médian de la planète de départ** du passager, puis 27 (la médiane de tout le monde) s'il ne reste un vide.

Résultat attendu : `0` dormeur dépensier, `119` éveillés déduits, aucune case vide dans `cryo_rempli` ni dans `age_rempli`.

<details><summary>Indice</summary>

Pour 3 : `vaisseau["CryoSleep"].map({True: 1, False: 0})` transforme en nombres (les vides restent vides), puis `.fillna((vaisseau["Depense"] == 0).astype(int))` remplit chaque vide avec 1 ou 0 selon la ligne.

Pour 4 : `vaisseau.groupby("HomePlanet")["Age"].transform("median")` donne, pour chaque ligne, l'âge médian de sa planète : passe-le à `.fillna(...)`, puis encore `.fillna(27)`.

</details>

In [ ]:
# À toi
nb_dormeurs_depensiers = None
nb_deduits_eveilles = None
cryo_rempli = None
age_rempli = None

print(nb_dormeurs_depensiers, "dormeur(s) dépensier(s),", nb_deduits_eveilles, "éveillés déduits")

In [ ]:
verifier("Exercice 5 · aucun dormeur ne dépense", lambda: nb_dormeurs_depensiers == 0)
verifier("Exercice 5 · éveillés déduits", lambda: nb_deduits_eveilles == 119)
verifier("Exercice 5 · cryo_rempli en 0/1, sans vide", lambda: cryo_rempli.isna().sum() == 0 and set(cryo_rempli.unique()) <= {0, 1} and cryo_rempli.sum() == 3135)
verifier("Exercice 5 · age_rempli sans vide, par planète", lambda: age_rempli.isna().sum() == 0
         and (age_rempli[vaisseau["Age"].isna() & (vaisseau["HomePlanet"] == "Europa")] == 33).all())

<details><summary>Solution</summary>

```python
nb_dormeurs_depensiers = ((vaisseau["CryoSleep"] == True) & (vaisseau["Depense"] > 0)).sum()
nb_deduits_eveilles = (vaisseau["CryoSleep"].isna() & (vaisseau["Depense"] > 0)).sum()
cryo_rempli = vaisseau["CryoSleep"].map({True: 1, False: 0}).fillna((vaisseau["Depense"] == 0).astype(int))
age_rempli = vaisseau["Age"].fillna(vaisseau.groupby("HomePlanet")["Age"].transform("median")).fillna(27)

print(nb_dormeurs_depensiers, "dormeur(s) dépensier(s),", nb_deduits_eveilles, "éveillés déduits")
print(vaisseau.groupby("HomePlanet")["Age"].median())   # Terre 23 ans, Mars 28, Europa 33
```

Zéro dormeur dépensier : la règle est sûre, on peut s'en servir pour remplir. C'est aussi pourquoi les exercices 3 et 4 donnaient presque la même chose : les dormeurs sont tous dans le groupe « rien dépensé ».

</details>

## Exercice 6 ⭐⭐ · La cabine cachée dans un texte (regex)

`Cabin` ressemble à `B/0/P` : le **pont** (une lettre), le **numéro** (des chiffres) et le **côté** (`P` = bâbord, `S` = tribord), séparés par des `/`. Avec une expression régulière :
- ajoute à `vaisseau` les colonnes `Pont` (la lettre) et `Cote` (`P` ou `S`)
- `taux_tribord` et `taux_babord` : la part de transportés de chaque côté
- `pont_le_plus_peuple` : la lettre du pont qui a le plus de passagers

Résultat attendu : environ `0.56` à tribord contre `0.45` à bâbord. L'anomalie est passée d'un côté du vaisseau.

<details><summary>Indice</summary>

Le motif `r"^(\w)/(\d+)/(\w)$"` a 3 groupes entre parenthèses : une lettre, des chiffres, une lettre. `vaisseau["Cabin"].str.extract(motif)` renvoie un tableau à 3 colonnes, numérotées 0, 1 et 2. Puis `groupby("Cote")`.

</details>

In [ ]:
# À toi
taux_tribord = None
taux_babord = None
pont_le_plus_peuple = None

print("Tribord :", taux_tribord, "- bâbord :", taux_babord, "- pont le plus peuplé :", pont_le_plus_peuple)

In [ ]:
verifier("Exercice 6 · colonnes Pont et Cote", lambda: set(vaisseau["Pont"].dropna()) <= set("ABCDEFGT") and set(vaisseau["Cote"].dropna()) == {"P", "S"})
verifier("Exercice 6 · tribord ≈ 0.56, bâbord ≈ 0.45", lambda: proche(taux_tribord, 0.555, 0.001) and proche(taux_babord, 0.4513, 0.001))
verifier("Exercice 6 · pont le plus peuplé", lambda: pont_le_plus_peuple == "F")

<details><summary>Solution</summary>

```python
cabine = vaisseau["Cabin"].str.extract(r"^(\w)/(\d+)/(\w)$")
vaisseau["Pont"] = cabine[0]
vaisseau["Cote"] = cabine[2]
par_cote = vaisseau.groupby("Cote")["Transported"].mean()
taux_tribord, taux_babord = par_cote["S"], par_cote["P"]
pont_le_plus_peuple = vaisseau["Pont"].value_counts().idxmax()

print("Tribord :", round(taux_tribord, 3), "- bâbord :", round(taux_babord, 3), "- pont le plus peuplé :", pont_le_plus_peuple)
print(vaisseau.groupby("Pont")["Transported"].mean().round(2))   # le pont B : 73 % de transportés
```

</details>

## Exercice 7 ⭐⭐ · Trois nouvelles variables : le groupe et le nom

`PassengerId` vaut par exemple `0003_02` : le deuxième passager du groupe 0003. Les membres d'un groupe voyagent ensemble, comme une famille sur le Titanic. Ajoute à `vaisseau` :
- `Groupe` : la partie avant le `_` (une chaîne, `"0003"`)
- `Taille_groupe` : combien de passagers ont le même `Groupe`
- `Seul` : 1 si `Taille_groupe` vaut 1, sinon 0
- `Nom` : le nom de famille, c'est-à-dire le dernier mot de `Name` (avec une regex)

Puis : `nb_seuls`, `taux_seuls` et `taux_en_groupe` (part de transportés de chaque côté), et `nom_le_plus_frequent`.

Résultat attendu : `4805` passagers seuls, transportés à environ `0.45` contre `0.57` en groupe.

<details><summary>Indice</summary>

`vaisseau["PassengerId"].str.split("_").str[0]` garde ce qui précède le `_`. Pour la taille : `vaisseau["Groupe"].map(vaisseau["Groupe"].value_counts())`, qui remplace chaque groupe par son nombre de passagers. Le dernier mot : `str.extract(r"(\S+)$")[0]` (`\S+` = des caractères qui ne sont pas des espaces, `$` = la fin du texte).

</details>

In [ ]:
# À toi
nb_seuls = None
taux_seuls = None
taux_en_groupe = None
nom_le_plus_frequent = None

print(nb_seuls, "seuls · seuls :", taux_seuls, "- en groupe :", taux_en_groupe, "· nom le plus fréquent :", nom_le_plus_frequent)

In [ ]:
verifier("Exercice 7 · Groupe et Taille_groupe", lambda: vaisseau.loc[0, "Groupe"] == "0001" and vaisseau["Taille_groupe"].max() == 8)
verifier("Exercice 7 · Seul", lambda: set(vaisseau["Seul"].unique()) <= {0, 1} and nb_seuls == 4805 == vaisseau["Seul"].sum())
verifier("Exercice 7 · seuls ≈ 0.45, en groupe ≈ 0.57", lambda: proche(taux_seuls, 0.4524, 0.001) and proche(taux_en_groupe, 0.5669, 0.001))
verifier("Exercice 7 · Nom", lambda: vaisseau.loc[0, "Nom"] == "Ofracculy" and nom_le_plus_frequent == "Casonston")

<details><summary>Solution</summary>

```python
vaisseau["Groupe"] = vaisseau["PassengerId"].str.split("_").str[0]
vaisseau["Taille_groupe"] = vaisseau["Groupe"].map(vaisseau["Groupe"].value_counts())
vaisseau["Seul"] = (vaisseau["Taille_groupe"] == 1).astype(int)
vaisseau["Nom"] = vaisseau["Name"].str.extract(r"(\S+)$")[0]

nb_seuls = vaisseau["Seul"].sum()
taux_seuls = vaisseau.loc[vaisseau["Seul"] == 1, "Transported"].mean()
taux_en_groupe = vaisseau.loc[vaisseau["Seul"] == 0, "Transported"].mean()
nom_le_plus_frequent = vaisseau["Nom"].value_counts().idxmax()

print(nb_seuls, "seuls · seuls :", round(taux_seuls, 3), "- en groupe :", round(taux_en_groupe, 3), "· nom le plus fréquent :", nom_le_plus_frequent)
```

Le nom de famille, lui, ne servira pas au modèle : 2 217 noms différents pour 8 493 passagers nommés, c'est un indice trop éparpillé. Le groupe dit déjà qui voyage avec qui.

</details>

## Exercice 8 ⭐⭐ · Ta version de preparer()

Range tout ce que tu viens de faire dans **une seule fonction**, comme dans la leçon : elle servira à l'identique sur le faux test de l'exercice 11. `preparer(df)` part d'une copie de `df` et renvoie un tableau de **nombres**, sans case vide, avec les 14 colonnes de `COLONNES` (déjà écrite dans la cellule) :

1. les dépenses vides valent 0, puis `Depense` = leur somme ;
2. `CryoSleep` en 0 / 1, rempli comme à l'exercice 5 ; `VIP` en 0 / 1, une case vide devient 0 ; `Age` rempli comme à l'exercice 5 ;
3. `Pont` (la lettre transformée en nombre, `"ABCDEFGT".index(lettre)`, une case vide devient `"F"`), `Cote` (1 pour tribord, 0 sinon) et `Taille_groupe` ;
4. `Planete` (depuis `HomePlanet`) et `Destination` en nombres, une case vide devient la valeur la plus fréquente.

Résultat attendu : `X = preparer(vaisseau)` a la forme `(8693, 14)` et aucune case vide.

<details><summary>Indice</summary>

Recopie les lignes des exercices 4 à 7 en remplaçant `vaisseau` par `d`. Pour transformer un texte en nombres : `d["HomePlanet"].fillna("Earth").map({"Earth": 0, "Europa": 1, "Mars": 2})`. Pour la lettre du pont : `cabine[0].fillna("F").map(lambda pont: "ABCDEFGT".index(pont))`.

</details>

In [ ]:
# À toi
DEPENSES = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
COLONNES = ["CryoSleep", "Age", "VIP", "Depense"] + DEPENSES + ["Planete", "Destination", "Pont", "Cote", "Taille_groupe"]

def preparer(df):
    """Transforme le tableau brut du vaisseau en tableau de nombres prêt pour un modèle."""
    d = df.copy()
    # 1. Les dépenses
    # 2. Les cases vides
    # 3. Les nouvelles variables
    # 4. Le texte en nombres
    return None    # ← à la fin : return d[COLONNES]


X = preparer(vaisseau)
y = vaisseau["Transported"].astype(int)
print(type(X))

In [ ]:
verifier("Exercice 8 · 14 colonnes, 8 693 lignes", lambda: X.shape == (8693, 14) and list(X.columns) == COLONNES)
verifier("Exercice 8 · aucune case vide", lambda: X.isna().sum().sum() == 0)
verifier("Exercice 8 · que des nombres", lambda: all(pd.api.types.is_numeric_dtype(X[col]) for col in X.columns))
verifier("Exercice 8 · les vides de CryoSleep remplis comme à l'exercice 5", lambda: X["CryoSleep"].sum() == 3135)

<details><summary>Solution</summary>

```python
DEPENSES = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
COLONNES = ["CryoSleep", "Age", "VIP", "Depense"] + DEPENSES + ["Planete", "Destination", "Pont", "Cote", "Taille_groupe"]

def preparer(df):
    """Transforme le tableau brut du vaisseau en tableau de nombres prêt pour un modèle."""
    d = df.copy()
    # 1. Les dépenses : une case vide compte 0 (la médiane de chaque colonne vaut 0)
    d[DEPENSES] = d[DEPENSES].fillna(0)
    d["Depense"] = d[DEPENSES].sum(axis=1)
    # 2. Les cases vides, avec un raisonnement
    d["CryoSleep"] = d["CryoSleep"].map({True: 1, False: 0}).fillna((d["Depense"] == 0).astype(int)).astype(int)
    d["VIP"] = d["VIP"].map({True: 1, False: 0}).fillna(0).astype(int)
    d["Age"] = d["Age"].fillna(d.groupby("HomePlanet")["Age"].transform("median")).fillna(27)
    # 3. Les nouvelles variables : la cabine (regex) et la taille du groupe
    cabine = d["Cabin"].str.extract(r"^(\w)/(\d+)/(\w)$")
    d["Pont"] = cabine[0].fillna("F").map(lambda pont: "ABCDEFGT".index(pont))
    d["Cote"] = (cabine[2] == "S").astype(int)
    groupe = d["PassengerId"].str.split("_").str[0]
    d["Taille_groupe"] = groupe.map(groupe.value_counts())
    # 4. Le texte en nombres
    d["Planete"] = d["HomePlanet"].fillna("Earth").map({"Earth": 0, "Europa": 1, "Mars": 2})
    d["Destination"] = d["Destination"].fillna("TRAPPIST-1e").map({"TRAPPIST-1e": 0, "55 Cancri e": 1, "PSO J318.5-22": 2})
    return d[COLONNES]


X = preparer(vaisseau)
y = vaisseau["Transported"].astype(int)
print(X.shape, "-", X.isna().sum().sum(), "case vide")
X.head()
```

</details>

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
try:
    _pret = isinstance(X, pd.DataFrame) and X.shape == (len(vaisseau), 14) and X.isna().sum().sum() == 0
except Exception:
    _pret = False
if not _pret:
    try:
        DEPENSES = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
        COLONNES = ["CryoSleep", "Age", "VIP", "Depense"] + DEPENSES + ["Planete", "Destination", "Pont", "Cote", "Taille_groupe"]

        def preparer(df):
            """Transforme le tableau brut du vaisseau en tableau de nombres prêt pour un modèle."""
            d = df.copy()
            # 1. Les dépenses : une case vide compte 0 (la médiane de chaque colonne vaut 0)
            d[DEPENSES] = d[DEPENSES].fillna(0)
            d["Depense"] = d[DEPENSES].sum(axis=1)
            # 2. Les cases vides, avec un raisonnement
            d["CryoSleep"] = d["CryoSleep"].map({True: 1, False: 0}).fillna((d["Depense"] == 0).astype(int)).astype(int)
            d["VIP"] = d["VIP"].map({True: 1, False: 0}).fillna(0).astype(int)
            d["Age"] = d["Age"].fillna(d.groupby("HomePlanet")["Age"].transform("median")).fillna(27)
            # 3. Les nouvelles variables : la cabine (regex) et la taille du groupe
            cabine = d["Cabin"].str.extract(r"^(\w)/(\d+)/(\w)$")
            d["Pont"] = cabine[0].fillna("F").map(lambda pont: "ABCDEFGT".index(pont))
            d["Cote"] = (cabine[2] == "S").astype(int)
            groupe = d["PassengerId"].str.split("_").str[0]
            d["Taille_groupe"] = groupe.map(groupe.value_counts())
            # 4. Le texte en nombres
            d["Planete"] = d["HomePlanet"].fillna("Earth").map({"Earth": 0, "Europa": 1, "Mars": 2})
            d["Destination"] = d["Destination"].fillna("TRAPPIST-1e").map({"TRAPPIST-1e": 0, "55 Cancri e": 1, "PSO J318.5-22": 2})
            return d[COLONNES]

        X = preparer(vaisseau)
        y = vaisseau["Transported"].astype(int)
        print("Filet : la recette de référence est chargée, X et y sont prêts pour la suite.")
    except Exception as erreur:
        print("Filet : impossible de préparer X →", erreur)

## Exercice 9 ⭐⭐ · Régression logistique sur 25 % cachés

Premier modèle, mesuré honnêtement :
- coupe `X` et `y` en deux avec `train_test_split(X, y, test_size=0.25, random_state=42)` → `X_app, X_val, y_app, y_val`
- `logistique` : une `LogisticRegression(solver="liblinear")` entraînée sur `X_app, y_app` (`liblinear` est un autre moteur de calcul, plus rapide sur ce tableau aux dépenses très étalées)
- `exactitude_logistique` : son exactitude sur `X_val, y_val`
- `exactitude_bete` : l'exactitude de la règle « tout le monde est transporté » sur les mêmes `y_val`

Résultat attendu : environ `0.77` pour le modèle, contre `0.50` pour la règle bête.

<details><summary>Indice</summary>

`modele.fit(X_app, y_app)` puis `modele.score(X_val, y_val)`. Pour la règle bête : elle prédit 1 partout, donc elle a raison chaque fois que `y_val` vaut 1 : `(y_val == 1).mean()`.

</details>

In [ ]:
# À toi
X_app, X_val, y_app, y_val = None, None, None, None
logistique = None
exactitude_logistique = None
exactitude_bete = None

print("Logistique :", exactitude_logistique, "- règle bête :", exactitude_bete)

In [ ]:
verifier("Exercice 9 · 25 % cachés", lambda: len(X_val) == 2174 and len(X_app) == 6519)
verifier("Exercice 9 · logistique ≈ 0.77", lambda: proche(exactitude_logistique, 0.7741, 0.01))
verifier("Exercice 9 · règle bête ≈ 0.50", lambda: proche(exactitude_bete, 0.5023, 0.001))

<details><summary>Solution</summary>

```python
X_app, X_val, y_app, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
logistique = LogisticRegression(solver="liblinear").fit(X_app, y_app)
exactitude_logistique = logistique.score(X_val, y_val)
exactitude_bete = (y_val == 1).mean()

print("Logistique :", round(exactitude_logistique, 3), "- règle bête :", round(exactitude_bete, 3))
```

</details>

## Exercice 10 ⭐⭐⭐ · Forêt aléatoire et validation croisée

Un seul découpage, c'est un seul contrôle blanc. Passe à la **validation croisée** en 5 paquets :
- `scores_logistique` : `cross_val_score(LogisticRegression(solver="liblinear"), X, y, cv=5)`
- `scores_foret` : la même chose avec `RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)` (une dizaine de secondes)
- `meilleur_modele` : `"forêt"` ou `"logistique"`, celui dont la **moyenne** des 5 scores est la plus haute

Résultat attendu : 5 scores par modèle, logistique ≈ `0.78` en moyenne, forêt ≈ `0.80`.

<details><summary>Indice</summary>

`cross_val_score` renvoie un tableau de 5 exactitudes : `.mean()` pour la moyenne, `.std()` pour la dispersion. Compare les deux moyennes.

</details>

In [ ]:
# À toi
scores_logistique = None
scores_foret = None
meilleur_modele = None

print("Logistique :", scores_logistique)
print("Forêt      :", scores_foret)
print("Meilleur   :", meilleur_modele)

In [ ]:
verifier("Exercice 10 · 5 scores par modèle", lambda: len(scores_logistique) == 5 and len(scores_foret) == 5)
verifier("Exercice 10 · logistique ≈ 0.78", lambda: proche(np.mean(scores_logistique), 0.7847, 0.01))
verifier("Exercice 10 · forêt ≈ 0.80", lambda: proche(np.mean(scores_foret), 0.7981, 0.01))
verifier("Exercice 10 · meilleur_modele", lambda: meilleur_modele.lower().startswith("for"))

<details><summary>Solution</summary>

```python
scores_logistique = cross_val_score(LogisticRegression(solver="liblinear"), X, y, cv=5)
foret = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
scores_foret = cross_val_score(foret, X, y, cv=5)
meilleur_modele = "forêt" if scores_foret.mean() > scores_logistique.mean() else "logistique"

print(f"Logistique : {scores_logistique.mean()*100:.1f} % (± {scores_logistique.std()*100:.1f})")
print(f"Forêt      : {scores_foret.mean()*100:.1f} % (± {scores_foret.std()*100:.1f})")
print("Meilleur   :", meilleur_modele)
```

</details>

## Exercice 11 ⭐⭐⭐ · Un faux test.csv et ta soumission

La préparation a mis de côté `faux_test` : 738 passagers des groupes 8500 et plus, **sans** la colonne `Transported`, comme un `test.csv` de Kaggle. Leurs vraies réponses sont dans `reponses_cachees` (0 / 1), et les 7 955 autres passagers forment `vaisseau_reduit`.

Fais exactement ce qu'on fait sur Kaggle :
1. Entraîne une forêt (`n_estimators=200, max_depth=8, random_state=42`) sur `preparer(vaisseau_reduit)` et `vaisseau_reduit["Transported"]`
2. Prédis `preparer(faux_test)` → `predictions`
3. Construis `submission` : un DataFrame à 2 colonnes, `PassengerId` et `Transported` (en `True` / `False`, le format de la compétition), et écris-le dans `submission_vaisseau.csv` (`index=False`)
4. `exactitude_faux_test` : la part de prédictions égales à `reponses_cachees`

Résultat attendu : un fichier de 738 lignes et 2 colonnes, une exactitude d'environ `0.80`.

<details><summary>Indice</summary>

`predictions.astype(bool)` transforme les 0 / 1 en `False` / `True`. Pour l'exactitude : `(predictions == reponses_cachees.values).mean()`. Le faux test est fait de **groupes entiers** : `Taille_groupe` s'y calcule donc aussi bien que dans un vrai `test.csv`.

</details>

In [ ]:
# À toi
modele = None
predictions = None
submission = None
exactitude_faux_test = None

print("Exactitude sur le faux test :", exactitude_faux_test)

In [ ]:
verifier("Exercice 11 · submission (DataFrame) et submission_vaisseau.csv existent", lambda: isinstance(submission, pd.DataFrame) and os.path.exists("submission_vaisseau.csv"))
verifier("Exercice 11 · 738 lignes, colonnes PassengerId et Transported",
         lambda: pd.read_csv("submission_vaisseau.csv").shape == (738, 2) and list(pd.read_csv("submission_vaisseau.csv").columns) == ["PassengerId", "Transported"])
verifier("Exercice 11 · prédictions en True / False", lambda: set(pd.read_csv("submission_vaisseau.csv")["Transported"].astype(str)) <= {"True", "False"})
verifier("Exercice 11 · exactitude_faux_test ≈ 0.80", lambda: proche(exactitude_faux_test, 0.8035, 0.02))

<details><summary>Solution</summary>

```python
modele = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
modele.fit(preparer(vaisseau_reduit), vaisseau_reduit["Transported"].astype(int))
predictions = modele.predict(preparer(faux_test))

submission = pd.DataFrame({"PassengerId": faux_test["PassengerId"], "Transported": predictions.astype(bool)})
submission.to_csv("submission_vaisseau.csv", index=False)
exactitude_faux_test = (predictions == reponses_cachees.values).mean()

print(submission.head())
print("Exactitude sur le faux test :", round(exactitude_faux_test, 3))
```

Sur Kaggle, c'est pareil avec le vrai `test.csv` : tu ne connais juste pas `reponses_cachees`.

</details>

## Exercice 12 ⭐⭐⭐ · Défi · Qui aurait été transporté ?

Écris `predire_transport(age, planete, dormeur, depense_spa=0.0, cabine="F/1/S")` qui renvoie la **probabilité** (entre 0 et 1) qu'un passager soit transporté, d'après `modele_complet` (une forêt entraînée sur tous les passagers, déjà écrite dans la cellule).

L'idée : fabriquer un tableau d'**une ligne** avec les mêmes colonnes **brutes** que `vaisseau` (sans `Transported`), le passer à `preparer()`, puis demander `modele_complet.predict_proba(...)[0][1]`.

Résultat attendu : une Européenne de 16 ans endormie dans une cabine du pont B est transportée à plus de 80 % ; un Terrien de 30 ans éveillé, qui a dépensé 3 000 au spa, à moins de 30 %.

<details><summary>Indice</summary>

Une ligne : `pd.DataFrame([{"PassengerId": "9999_01", "HomePlanet": planete, "CryoSleep": dormeur, "Cabin": cabine, "Destination": "TRAPPIST-1e", "Age": age, "VIP": False, "RoomService": 0.0, "FoodCourt": 0.0, "ShoppingMall": 0.0, "Spa": depense_spa, "VRDeck": 0.0, "Name": "Moi Meme"}])`.

</details>

In [ ]:
# À toi
modele_complet = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42).fit(X, y)

def predire_transport(age, planete, dormeur, depense_spa=0.0, cabine="F/1/S"):
    """planete : "Earth", "Europa" ou "Mars" ; dormeur : True ou False. Renvoie une probabilité."""
    return None


proba_europeenne = predire_transport(16, "Europa", True, cabine="B/12/S")
proba_terrien = predire_transport(30, "Earth", False, depense_spa=3000.0, cabine="G/800/P")
print("Européenne endormie :", proba_europeenne, "- Terrien au spa :", proba_terrien)

In [ ]:
verifier("Exercice 12 · la fonction renvoie une probabilité", lambda: 0 <= float(proba_europeenne) <= 1 and 0 <= float(proba_terrien) <= 1)
verifier("Exercice 12 · l'Européenne endormie est transportée (> 0.8)", lambda: float(proba_europeenne) > 0.8)
verifier("Exercice 12 · le Terrien au spa ne l'est pas (< 0.3)", lambda: float(proba_terrien) < 0.3)

<details><summary>Solution</summary>

```python
modele_complet = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42).fit(X, y)

def predire_transport(age, planete, dormeur, depense_spa=0.0, cabine="F/1/S"):
    """planete : "Earth", "Europa" ou "Mars" ; dormeur : True ou False. Renvoie une probabilité."""
    passager = pd.DataFrame([{
        "PassengerId": "9999_01", "HomePlanet": planete, "CryoSleep": dormeur, "Cabin": cabine,
        "Destination": "TRAPPIST-1e", "Age": age, "VIP": False, "RoomService": 0.0, "FoodCourt": 0.0,
        "ShoppingMall": 0.0, "Spa": depense_spa, "VRDeck": 0.0, "Name": "Moi Meme",
    }])
    return modele_complet.predict_proba(preparer(passager))[0][1]


proba_europeenne = predire_transport(16, "Europa", True, cabine="B/12/S")
proba_terrien = predire_transport(30, "Earth", False, depense_spa=3000.0, cabine="G/800/P")
print(f"Européenne endormie : {proba_europeenne*100:.0f} % - Terrien au spa : {proba_terrien*100:.0f} %")
# Environ 97 % contre 11 % : le sommeil et les dépenses font presque tout le travail.
```

</details>

## Bravo !

Tu as refait seul·e tout le pipeline de la séance, sur des données que la leçon n'a pas vues : explorer, vérifier des hypothèses, remplir les cases vides avec un raisonnement, extraire des variables d'un texte avec une regex, ranger la recette dans `preparer()`, mesurer honnêtement et produire un fichier de soumission.

**Pour aller plus loin** : le *Spaceship Titanic* est une vraie compétition Kaggle, ouverte en permanence : https://www.kaggle.com/competitions/spaceship-titanic. Rejoins-la, dépose son `test.csv` dans Colab, remplace `faux_test` par `pd.read_csv("test.csv")` dans l'exercice 11, et soumets `submission_vaisseau.csv`.